# Fashion-MNIST CNN Classification
## Practical Task 1: CNN for Fashion-MNIST Classification (10 Marks)

**Objective:** Implement a Convolutional Neural Network to classify images from the Fashion-MNIST dataset.

**Dataset:** Fashion-MNIST contains 70,000 grayscale images (28x28 pixels) across 10 fashion categories.

## 1. Import Required Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow version: {tf.__version__}")

## 2. Load and Explore the Fashion-MNIST Dataset

In [ ]:
# Load the dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Class names for Fashion-MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Visualize sample images
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i], cmap='gray')
    plt.xlabel(class_names[y_train[i]])
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape data to add channel dimension (28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Preprocessed training data shape: {X_train.shape}")
print(f"Preprocessed test data shape: {X_test.shape}")
print(f"One-hot encoded labels shape: {y_train_cat.shape}")

## 4. Build the CNN Model

### Architecture Choices:
- **Conv Layer 1:** 32 filters, 3x3 kernel - captures basic features like edges
- **Conv Layer 2:** 64 filters, 3x3 kernel - captures more complex patterns
- **MaxPooling:** 2x2 - reduces spatial dimensions, provides translation invariance
- **Dropout:** 0.25 after pooling, 0.5 after dense - prevents overfitting
- **Dense Layer:** 128 units - learns high-level representations
- **Output Layer:** 10 units with softmax - multi-class classification

In [ ]:
def create_cnn_model():
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        
        # Output Layer
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Create the model
model = create_cnn_model()
model.summary()

## 5. Compile the Model

In [ ]:
# Compile with Adam optimizer and categorical crossentropy loss
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 6. Train the Model

In [ ]:
# Train for 15 epochs with validation split
history = model.fit(
    X_train, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

## 7. Evaluate on Test Set

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## 8. Plot Training History (Accuracy and Loss Curves)

In [ ]:
# Plot accuracy curves
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss curves
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. Generate Predictions and Confusion Matrix

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Classification Report (Precision, Recall, F1-Score)

In [ ]:
# Generate classification report
print("\nClassification Report:")
print("=" * 70)
report = classification_report(y_test, y_pred_classes, target_names=class_names)
print(report)

## 11. Visualize Sample Predictions

In [ ]:
# Visualize some predictions
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    
    predicted_label = y_pred_classes[i]
    true_label = y_test[i]
    color = 'green' if predicted_label == true_label else 'red'
    
    plt.xlabel(f"Pred: {class_names[predicted_label]}\nTrue: {class_names[true_label]}", 
               color=color)
plt.tight_layout()
plt.show()

## 12. Analysis and Discussion

### Architecture Choices Explanation:

1. **Two Convolutional Layers:**
   - First layer (32 filters): Detects low-level features like edges and textures
   - Second layer (64 filters): Captures more complex patterns and combinations
   - 3x3 kernel size: Standard choice, good balance between receptive field and computation

2. **MaxPooling Layers (2x2):**
   - Reduces spatial dimensions by half
   - Provides translation invariance
   - Reduces computational cost

3. **Dropout Regularization:**
   - 0.25 after conv blocks: Mild regularization
   - 0.5 after dense layer: Stronger regularization where overfitting is more likely

4. **Dense Layer (128 units):**
   - Learns high-level feature combinations
   - ReLU activation for non-linearity

5. **Output Layer:**
   - 10 units (one per class)
   - Softmax activation for probability distribution

### Performance Analysis:

**Expected Results:**
- Test accuracy should be around 90-92%
- Training should converge within 10-15 epochs

**Overfitting Assessment:**
- Check if validation loss increases while training loss decreases
- Compare training vs validation accuracy gap
- If gap > 5%, model is overfitting

**Potential Improvements:**
1. **Data Augmentation:** Random rotations, shifts, zooms
2. **Batch Normalization:** Add after conv layers for faster convergence
3. **Learning Rate Scheduling:** Reduce LR when plateau
4. **More Conv Layers:** Add a third conv block with 128 filters
5. **Early Stopping:** Stop training when validation loss stops improving
6. **Ensemble Methods:** Combine multiple models
7. **Transfer Learning:** Use pre-trained models (though less common for Fashion-MNIST)

## 13. Save the Model (Optional)

In [ ]:
# Save the trained model
model.save('fashion_mnist_cnn_model.h5')
print("Model saved successfully!")